# Notebook16 sampled hierarchical kernel operator
Adaptive cross approximation forms sampled rows and columns in well separated blocks and dense matrices only at near leaves. Local residual and negative-entry screens are empirical; they do not prove uniform kernel error or global positivity of approximate entries. Pivot/topology choices are detached, so gradients are piecewise derivatives of chosen factors. Independent frozen training-image full output and parameter-gradient audits gate promotion.
Inputs: raw learned Q/K/V, masks and spatial coordinates. Outputs: normalized interactions and work/error diagnostics. Definitions perform no training. Exact resume uses source-bound parent16 full-state checkpoints.

In [ ]:
"""Differentiable sampled-entry adaptive block hierarchy; dense matrices only at near leaves."""
from dataclasses import dataclass
import copy,math,time
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

@dataclass
class Cluster:
    ids:object
    lo:object
    hi:object
    children:object


def cluster_tree(positions,leaf_size=32):
    p=positions.detach().double().cpu().numpy();leaf_size=int(leaf_size)
    if leaf_size<2 or p.ndim!=2 or p.shape[1]!=2:raise ValueError('Two-dimensional positions and leaf size>=2 required')
    def build(ids):
        lo=p[ids].min(0);hi=p[ids].max(0);children=[]
        if len(ids)>leaf_size:
            axis=int(np.argmax(hi-lo));order=ids[np.lexsort((p[ids,1-axis],p[ids,axis]))];middle=len(order)//2
            children=[build(order[:middle]),build(order[middle:])]
        return Cluster(ids,lo,hi,children)
    return build(np.arange(len(p),dtype=np.int64))


def block_partition(left,right,eta=1.):
    gap=np.maximum(0.,np.maximum(left.lo-right.hi,right.lo-left.hi));distance=float(np.linalg.norm(gap))
    diameter=max(float(np.linalg.norm(left.hi-left.lo)),float(np.linalg.norm(right.hi-right.lo)))
    if distance>float(eta)*diameter:return [('far',left,right)]
    if not left.children and not right.children:return [('near',left,right)]
    if left.children and (not right.children or len(left.ids)>=len(right.ids)):
        return sum((block_partition(child,right,eta)for child in left.children),[])
    return sum((block_partition(left,child,eta)for child in right.children),[])


def _split_block(left,right,eta):
    if left.children and (not right.children or len(left.ids)>=len(right.ids)):
        return sum((block_partition(child,right,eta)for child in left.children),[])
    if right.children:return sum((block_partition(left,child,eta)for child in right.children),[])
    return []


def _relative(a,b):
    return float((a.detach()-b.detach()).norm()/b.detach().norm().clamp_min(torch.finfo(b.dtype).tiny))


def _aca(entry,m,n,max_rank,tolerance,stats,probe_count=8):
    """Adaptive cross approximation evaluates rows/columns and a small rank-selection grid only.

    Pivot choices are detached discrete controls. The chosen sampled matrix
    entries/factors remain differentiable; gradient accuracy is tested separately.
    No full far block is formed or factorized.
    """
    device=entry.device;dtype=entry.dtype
    if m<=probe_count and n<=probe_count:
        stats['rejected_far_blocks']+=1
        return None  # Small FAR leaves use the explicitly counted dense fallback.
    row_probe=torch.unique(torch.linspace(0,m-1,min(probe_count,m),device=device).round().long())
    col_probe=torch.unique(torch.linspace(0,n-1,min(probe_count,n),device=device).round().long())
    test=entry(row_probe,col_probe);stats['sampled_kernel_entries']+=len(row_probe)*len(col_probe)
    us=[];vs=[];used=set();pivot_row=0;error=float('inf')
    all_rows=torch.arange(m,device=device);all_cols=torch.arange(n,device=device)
    for rank in range(min(int(max_rank),m,n)):
        row=entry(torch.tensor([pivot_row],device=device),all_cols).squeeze(0)
        if us:row=row-torch.stack([u[pivot_row]for u in us])@torch.stack(vs)
        pivot_col=int(row.detach().abs().argmax());pivot=row[pivot_col]
        column=entry(all_rows,torch.tensor([pivot_col],device=device)).squeeze(1)
        if us:column=column-torch.stack(us,1)@torch.stack([v[pivot_col]for v in vs])
        stats['sampled_kernel_entries']+=m+n
        magnitude=float(test.detach().abs().max())
        if float(pivot.detach().abs())<=torch.finfo(dtype).eps*max(magnitude,torch.finfo(dtype).tiny):break
        u=column/pivot;v=row;us.append(u);vs.append(v);used.add(pivot_row)
        U=torch.stack(us,1);V=torch.stack(vs)
        approximation=U.index_select(0,row_probe)@V.index_select(1,col_probe)
        error=_relative(approximation,test)
        negative=bool((approximation.detach() < -32*torch.finfo(dtype).eps*test.detach().abs().max()).any())
        if negative:stats['negative_sample_rejections']+=1
        if error<=float(tolerance) and not negative:
            stats['far_blocks']+=1;stats['factor_elements']+=U.numel()+V.numel();stats['far_ranks'].append(len(us));stats['sampled_residuals'].append(error)
            return U,V
        candidates=column.detach().abs().clone()
        if used:candidates[list(used)]=-1
        pivot_row=int(candidates.argmax())
        if float(candidates[pivot_row])<0:break
    stats['rejected_far_blocks']+=1
    return None


def hierarchical_attention(q,k,v,valid,positions,tau=None,spatial_coefficient=None,spatial_length=None,
                           leaf_size=32,max_rank=16,tolerance=1e-4,eta=1.,return_stats=False):
    """Approximate an exponential score kernel with matching numerator/denominator.

    Only sampled factor entries are screened for positivity; no global
    positivity or uniform approximation-error certificate is asserted.

    q/k/v: [B,H,N,D]. valid: [B,N]. positions: [B,N,2]. Scores are
    q_i·k_j/tau_h − lambda_h||p_i−p_j||²/(2ell_h²).
    Invalid queries return zero, invalid sources never enter any block.
    A row-wise Cauchy bound prevents exponential overflow. If this bound is
    numerically too loose, the operator rejects the input rather than fabricate
    a denominator; convergence/error gates must handle that failure explicitly.
    """
    if q.shape!=k.shape or q.shape!=v.shape or q.ndim!=4:raise ValueError('Matching [B,H,N,D] q/k/v required')
    B,H,N,D=q.shape
    if valid.shape!=(B,N)or positions.shape!=(B,N,2)or valid.dtype!=torch.bool:raise ValueError('Mask/position shape differs')
    if not all(torch.isfinite(x).all()for x in(q,k,v,positions)):raise ValueError('Finite inputs required')
    def heads(value,default):
        return torch.full((H,),float(default),device=q.device,dtype=q.dtype)if value is None else torch.as_tensor(value,device=q.device,dtype=q.dtype).reshape(H)
    tau=heads(tau,math.sqrt(D));lam=heads(spatial_coefficient,0.);ell=heads(spatial_length,.35)
    if bool((tau<=0).any())or bool((ell<=0).any()):raise ValueError('Positive temperature/length required')
    stats={'algorithm':'sampled-entry adaptive cross block hierarchy','dense_entries':0,'sampled_kernel_entries':0,
           'factor_elements':0,'far_blocks':0,'near_blocks':0,'rejected_far_blocks':0,'uncompressed_far_leaf_blocks':0,
           'negative_sample_rejections':0,'entry_positivity_scope':'Only sampled ACA entries screened; no global approximate-entry positivity guarantee',
           'error_scope':'Empirical local ACA rank selection; independent learned-token output/gradient audits required, no uniform kernel certificate',
           'far_ranks':[],'sampled_residuals':[],'full_far_matrix_constructed':False,'full_far_matrix_scope':'No non-leaf FAR matrix; explicitly counted small FAR leaves may use dense fallback','dense_matrix_factorization_used':False,
           'largest_dense_block_elements':0,'dense_reference_entries':B*H*N*N,'gradient_scope':'Differentiable sampled factors; detached pivots/topology. Accuracy gates compare Q/K/V and geometry gradients.'}
    batches=[]
    for b in range(B):
        ids=torch.nonzero(valid[b],as_tuple=False).flatten()
        if not len(ids):batches.append(q[b]*0.+k[b]*0.+v[b]*0.);continue
        pos=positions[b].index_select(0,ids);tree=cluster_tree(pos,leaf_size);initial=block_partition(tree,tree,eta)
        outputs=[]
        for h in range(H):
            Q=q[b,h].index_select(0,ids);K=k[b,h].index_select(0,ids);V=v[b,h].index_select(0,ids);count=len(ids)
            span=(pos.max(0).values-pos.min(0).values).square().sum()
            shift=(Q.norm(dim=-1)*K.norm(dim=-1).max()/tau[h]+(-lam[h]).clamp_min(0.)*span/(2*ell[h].square())).detach()
            numerator=torch.zeros_like(V);denominator=torch.zeros(count,device=q.device,dtype=q.dtype)
            queue=list(initial)
            while queue:
                kind,left,right=queue.pop()
                ri=torch.as_tensor(left.ids,device=q.device);ci=torch.as_tensor(right.ids,device=q.device)
                qr=Q.index_select(0,ri);kc=K.index_select(0,ci);pr=pos.index_select(0,ri);pc=pos.index_select(0,ci);sr=shift.index_select(0,ri)
                def entry(rr,cc):
                    qa=qr.index_select(0,rr);ka=kc.index_select(0,cc);pa=pr.index_select(0,rr);pb=pc.index_select(0,cc)
                    distance=(pa[:,None,:]-pb[None,:,:]).square().sum(-1)
                    score=qa@ka.T/tau[h]-lam[h]*distance/(2*ell[h].square())-sr.index_select(0,rr)[:,None]
                    return torch.exp(score)
                entry.device=q.device;entry.dtype=q.dtype
                factors=_aca(entry,len(ri),len(ci),max_rank,tolerance,stats)if kind=='far'else None
                if kind=='far'and factors is None:
                    children=_split_block(left,right,eta)
                    if children:queue.extend(children);continue
                    stats['uncompressed_far_leaf_blocks']+=1
                if factors is None:
                    if len(ri)>leaf_size or len(ci)>leaf_size:raise AssertionError('Dense construction allowed only at leaves')
                    matrix=entry(torch.arange(len(ri),device=q.device),torch.arange(len(ci),device=q.device))
                    stats['dense_entries']+=matrix.numel();stats['near_blocks']+=1;stats['largest_dense_block_elements']=max(stats['largest_dense_block_elements'],matrix.numel())
                    num=matrix@V.index_select(0,ci);den=matrix.sum(1)
                else:
                    U,W=factors;num=U@(W@V.index_select(0,ci));den=U@W.sum(1)
                numerator=numerator.index_copy(0,ri,numerator.index_select(0,ri)+num)
                denominator=denominator.index_copy(0,ri,denominator.index_select(0,ri)+den)
            if not torch.isfinite(denominator).all()or bool((denominator<=torch.finfo(q.dtype).tiny).any()):raise FloatingPointError('Approximate kernel has unsupported nonpositive/underflowed normalization')
            out=numerator/denominator[:,None];full=torch.zeros_like(v[b,h]);full=full.index_copy(0,ids,out);outputs.append(full)
        batches.append(torch.stack(outputs))
    output=torch.stack(batches)
    stats['visited_kernel_entries']=stats['dense_entries']+stats['sampled_kernel_entries']
    stats['entry_work_over_dense']=stats['visited_kernel_entries']/max(1,stats['dense_reference_entries'])
    stats['accepted_far_rank_max']=max(stats['far_ranks'],default=0)
    return (output,stats)if return_stats else output


def exact_attention(q,k,v,valid,positions,tau=None,spatial_coefficient=None,spatial_length=None):
    B,H,N,D=q.shape;scale=torch.full((H,),math.sqrt(D),device=q.device,dtype=q.dtype)if tau is None else tau
    lam=torch.zeros(H,device=q.device,dtype=q.dtype)if spatial_coefficient is None else spatial_coefficient
    ell=torch.full((H,),.35,device=q.device,dtype=q.dtype)if spatial_length is None else spatial_length
    scores=torch.einsum('bhid,bhjd->bhij',q,k)/scale[None,:,None,None]
    distance=(positions[:,:,None,:]-positions[:,None,:,:]).square().sum(-1)
    scores=scores-lam[None,:,None,None]*distance[:,None]/(2*ell[None,:,None,None].square())
    scores=scores.masked_fill(~valid[:,None,None,:],-torch.inf)
    scores=torch.where(valid.any(-1)[:,None,None,None],scores,torch.zeros_like(scores))
    weights=torch.nan_to_num(torch.softmax(scores,dim=-1));return (weights@v)*valid[:,None,:,None]


def sdpa_attention(q,k,v,valid,positions,tau=None,spatial_coefficient=None,spatial_length=None):
    B,H,N,D=q.shape;root=math.sqrt(D);tau=torch.full((H,),root,device=q.device,dtype=q.dtype)if tau is None else tau
    scaled=q*(root/tau)[None,:,None,None]
    if spatial_coefficient is None:
        # No dense additive mask for pure attention; expose optimized SDPA dispatch.
        mask=valid[:,None,None,:]
    else:
        lam=spatial_coefficient;ell=spatial_length
        distance=(positions[:,:,None,:]-positions[:,None,:,:]).square().sum(-1)
        mask=-lam[None,:,None,None]*distance[:,None]/(2*ell[None,:,None,None].square())
        mask=mask.masked_fill(~valid[:,None,None,:],-torch.inf)
    output=F.scaled_dot_product_attention(scaled,k,v,attn_mask=mask,dropout_p=0.,is_causal=False)
    return torch.nan_to_num(output)*valid[:,None,:,None]


class HierarchicalMixer(nn.Module):
    """Same learned parameters/projections as15; replace only the interaction application."""
    def __init__(self,reference,config=None):
        super().__init__();self.dim=reference.dim;self.heads=reference.heads;self.hierarchy_config=dict(config or{})
        for name,module in reference.named_children():self.add_module(name,copy.deepcopy(module))
        for name,param in reference._parameters.items():
            if param is not None:self.register_parameter(name,nn.Parameter(param.detach().clone(),requires_grad=param.requires_grad))
        for name,value in reference._buffers.items():
            if value is not None:self.register_buffer(name,value.detach().clone())
        self.spatial_coefficient_bound=getattr(reference,'spatial_coefficient_bound',2.);self.coefficient_bound=self.spatial_coefficient_bound;self.last_operator_stats={}
    def geometry_parameters(self,dtype=None):
        dtype=dtype or self.qkv.weight.dtype;device=self.qkv.weight.device;d=self.dim//self.heads
        mult=self.log_temperature_multiplier.clamp(-math.log(16),math.log(16)).exp() if hasattr(self,'log_temperature_multiplier') else torch.ones(self.heads,device=device,dtype=dtype)
        lam=self.spatial_coefficient_bound*self.spatial_coefficient_raw.tanh() if hasattr(self,'spatial_coefficient_raw') else torch.zeros(self.heads,device=device,dtype=dtype)
        ell=self.log_spatial_length.clamp(math.log(.03),math.log(2.)).exp() if hasattr(self,'log_spatial_length') else torch.full((self.heads,),.35,device=device,dtype=dtype)
        return {'temperature':(math.sqrt(d)*mult).to(dtype),'temperature_multiplier':mult.to(dtype),'spatial_coefficient':lam.to(dtype),'spatial_length':ell.to(dtype)}
    def kernel_state(self,x,indices,valid,grid):
        """Exact TARGET diagnostic, never the approximate interaction used in forward."""
        B,N,D=x.shape;safe=torch.where(valid[...,None],x,torch.zeros_like(x))
        q,k,v=self.qkv(safe).reshape(B,N,3,self.heads,D//self.heads).permute(2,0,3,1,4).unbind(0)
        pos=torch.stack(((indices//grid[1]+.5)/grid[0],(indices%grid[1]+.5)/grid[1]),-1).to(x.dtype);geo=self.geometry_parameters(x.dtype)
        distance=(pos[:,:,None]-pos[:,None,:]).square().sum(-1)
        scores=(q@k.transpose(-2,-1))/geo['temperature'][None,:,None,None]-geo['spatial_coefficient'][None,:,None,None]*distance[:,None]/(2*geo['spatial_length'][None,:,None,None].square())
        masked=scores.masked_fill(~valid[:,None,None,:],-torch.inf);masked=torch.where(valid.any(-1)[:,None,None,None],masked,torch.zeros_like(masked))
        weights=masked.softmax(-1)*valid[:,None,:,None]
        return {'q':q,'k':k,'v':v,'query':q,'key':k,'value':v,'weights':weights,'scores':scores,'logits':scores,'geometry':'Exact raw64 optional signed spatial TARGET kernel diagnostic; approximate interaction audited separately','positions':pos,**geo,'diagnostic_scope':'EXACT_TARGET_KERNEL_ONLY; hierarchical approximation separately audited'}
    def forward(self,x,indices=None,valid=None,grid=None):
        B,N,D=x.shape
        if valid is None:valid=torch.ones((B,N),device=x.device,dtype=torch.bool)
        if indices is None:indices=torch.arange(N,device=x.device).expand(B,-1)
        if grid is None:
            side=int(math.ceil(math.sqrt(N)));grid=(side,side)
        safe=torch.where(valid[...,None],x,torch.zeros_like(x))
        packed=self.qkv(safe).reshape(B,N,3,self.heads,D//self.heads).permute(2,0,3,1,4);q,k,v=packed.unbind(0)
        pos=torch.stack(((indices//grid[1]+.5)/grid[0],(indices%grid[1]+.5)/grid[1]),-1).to(x.dtype)
        tau=math.sqrt(D//self.heads)*self.log_temperature_multiplier.clamp(-math.log(16),math.log(16)).exp()if hasattr(self,'log_temperature_multiplier')else None
        lam=self.coefficient_bound*self.spatial_coefficient_raw.tanh()if hasattr(self,'spatial_coefficient_raw')else None
        ell=self.log_spatial_length.clamp(math.log(.03),math.log(2.)).exp()if hasattr(self,'log_spatial_length')else None
        out,stats=hierarchical_attention(q,k,v,valid,pos,tau,lam,ell,return_stats=True,**self.hierarchy_config);self.last_operator_stats=stats
        out=out.transpose(1,2).reshape(B,N,D);return self.projection(out)*valid[...,None]


def replace_encoder_mixers(model,config=None):
    for encoder in(model.context,model.target):
        for block in encoder.blocks:block.mixer=HierarchicalMixer(block.mixer,config)
    return model
